# USS Hustle on Google Colab
Explore the n8n API catalog (`skills/n8n/catalog.json`) from Colab. Add `N8N_API_KEY` and `N8N_BASE_URL` under Colab **Secrets**; nothing is hardcoded. Replace `REPO_URL` with this repo's clone URL if needed.

In [ ]:
REPO_URL = 'https://github.com/b2wbullpen/ai_studio.git'
BRANCH = 'copilot/create-ai-studio-project'
!git clone -q --branch $BRANCH $REPO_URL ai_studio 2>/dev/null || true
%cd ai_studio

In [ ]:
import json, os
from google.colab import userdata
os.environ['N8N_API_KEY'] = userdata.get('N8N_API_KEY')
os.environ['N8N_BASE_URL'] = userdata.get('N8N_BASE_URL')
catalog = json.load(open('skills/n8n/catalog.json'))
for e in catalog['endpoints']:
    print(e['method'], e['path'], '-', e['description'])

In [ ]:
import requests
def call(name, **params):
    ep = next(e for e in catalog['endpoints'] if e['name'] == name)
    path = ep['path']
    for k in [p['name'] for p in ep.get('path_params', [])]:
        path = path.replace(':' + k, str(params.pop(k)))
    r = requests.request(ep['method'], os.environ['N8N_BASE_URL'].rstrip('/') + path,
                         headers={catalog['auth']['header_name']: os.environ['N8N_API_KEY']},
                         params=params, timeout=30)
    r.raise_for_status()
    return r.json()

call('workflows_list', limit=10)